In [1]:
import pandas as pd
import numpy as np
import h3
import heapq
from collections import defaultdict

# --- Configuration ---
FILE_PATH = "./df.parquet"
OUTPUT_PATH = "./output/batchable_dynamic_results.csv"
H3_RES = 9  # Micro-hex for exact matching (~174m edge)
BASE_WAIT_MINS = 3

In [2]:
print("Loading data and computing base features...")
df = pd.read_parquet(FILE_PATH)

# Precompute time bins and temporal features
df["ts"] = pd.to_datetime(df["ts"])
df["ts_sec"] = df["ts"].astype("int64") // 10**9
df["t_bin"] = df["ts_sec"] // 60
df["hour"] = df["ts"].dt.hour

# Compute H9 Hexes
df["pickup_hex"] = [
    h3.latlng_to_cell(lat, lon, H3_RES) 
    for lat, lon in zip(df["trip_start_lat"], df["trip_start_lon"])
]
df["drop_hex"] = [
    h3.latlng_to_cell(lat, lon, H3_RES) 
    for lat, lon in zip(df["trip_end_lat"], df["trip_end_lon"])
]

# Calculate Raw ETA proxy
df["driver_arrival_time"] = pd.to_datetime(df["driver_arrival_time"])
df["arrival_proxy"] = df["driver_arrival_time"].fillna(pd.to_datetime(df["trip_start_time"]))
df["eta_seconds"] = (df["arrival_proxy"] - df["ts"]).dt.total_seconds().clip(lower=0)

Loading data and computing base features...


In [3]:
print("Calculating smoothed neighborhood demand and supply...")

# Aggregate raw stats for every individual H9 hex per hour
hex_stats = df.groupby(["pickup_hex", "hour"]).agg(
    raw_demand=("rid", "count"),
    raw_eta=("eta_seconds", "median")
).reset_index()

demand_dict = hex_stats.set_index(["pickup_hex", "hour"])["raw_demand"].to_dict()
eta_dict = hex_stats.set_index(["pickup_hex", "hour"])["raw_eta"].to_dict()

# Apply smoothing only to unique contexts to save massive CPU time
def get_smoothed_stats(h9_hex, hour):
    neighbors = h3.grid_disk(h9_hex, 1) # 1-ring neighborhood (~0.7 sq km)
    total_demand = 0
    etas = []
    
    for n in neighbors:
        key = (n, hour)
        total_demand += demand_dict.get(key, 0)
        eta_val = eta_dict.get(key)
        if pd.notnull(eta_val):
            etas.append(eta_val)
            
    median_eta = np.median(etas) if etas else 0
    return total_demand, median_eta

unique_contexts = df[["pickup_hex", "hour"]].drop_duplicates().copy()
smoothed_results = [
    get_smoothed_stats(row.pickup_hex, row.hour) 
    for _, row in unique_contexts.iterrows()
]

unique_contexts["neighbor_demand"] = [res[0] for res in smoothed_results]
unique_contexts["neighbor_eta"] = [res[1] for res in smoothed_results]

# Merge the smoothed neighborhood metrics back to the main dataset
df = df.merge(unique_contexts, on=["pickup_hex", "hour"], how="left")

Calculating smoothed neighborhood demand and supply...


In [4]:
print("Applying business rules for dynamic wait times...")

# Distance Factor: +1 min per 5km (capped at +4)
distance_factor = (df["chargeable_distance"] / 5000).fillna(0).clip(upper=4.0)

# Peak Time Factor: -1 min during rush hours (8-10 AM, 5-8 PM)
peak_hours = [8, 9, 10, 17, 18, 19]
time_factor = np.where(df["hour"].isin(peak_hours), -1.0, 0.0)

# Crowd Density Factor: >50 requests/hr = busy (-1m), <10 = quiet (+2m)
density_factor = np.where(
    df["neighbor_demand"] > 50, -1.0,   
    np.where(df["neighbor_demand"] < 10, 2.0, 0.0)  
)

# Supply Proxy Factor: ETA > 10m = low supply (+2m), ETA < 3m = high supply (-1m)
supply_factor = np.where(
    df["neighbor_eta"] > 600, 2.0,   
    np.where(df["neighbor_eta"] < 180, -1.0, 0.0)  
)

# Combine, clamp between 2 and 10 minutes, and assign
raw_wait_time = BASE_WAIT_MINS + distance_factor + time_factor + density_factor + supply_factor
df["max_wait_mins"] = raw_wait_time.clip(lower=3.0, upper=8.0).astype(int)

# Cleanup heavy columns
df = df.drop(columns=["hour", "arrival_proxy", "eta_seconds", "neighbor_demand", "neighbor_eta"])

Applying business rules for dynamic wait times...


In [5]:
print("Running the relaxed batching engine...")

# --- NEW TUNING PARAMETERS ---
PICKUP_RING_SIZE = 1  # Strict: ~170m radius (Easy walking distance)
DROP_RING_SIZE = 1   # Relaxed: ~800m radius (Short 2-3 min driving detour)

# Extract only what's needed for the matching algorithm
rides = df[["rid", "pickup_hex", "drop_hex", "t_bin", "ts", "max_wait_mins"]].copy()
rides["orig_idx"] = df.index
rides = rides.sort_values("t_bin").reset_index(drop=True)
rides["is_batchable"] = False
rides["batch_id"] = -1

# Precompute spatial neighbors with our new relaxed drop-off radius
pickup_neighbors_map = {h: set(h3.grid_disk(h, PICKUP_RING_SIZE)) for h in rides["pickup_hex"].unique()}
drop_neighbors_map = {h: set(h3.grid_disk(h, DROP_RING_SIZE)) for h in rides["drop_hex"].unique()}

buffer = []  # Min-Heap to hold tuples of (expiry_time, index)
pickup_index = defaultdict(set)  # Maps pickup_hex to active buffer indices
batch_counter = 0

for i in range(len(rides)):
    current_time = rides.at[i, "t_bin"]
    
    # 1. Expire old rides from the queue based on their specific max wait time
    while buffer and buffer[0][0] < current_time:
        expiry_time, old_idx = heapq.heappop(buffer)
        if not rides.at[old_idx, "is_batchable"]:
            old_pickup_hex = rides.at[old_idx, "pickup_hex"]
            pickup_index[old_pickup_hex].discard(old_idx)
            
    if rides.at[i, "is_batchable"]:
        continue
        
    # 2. Search for a spatial match
    pickup_i = rides.at[i, "pickup_hex"]
    drop_i = rides.at[i, "drop_hex"]
    
    matched = False
    
    for neighbor_pickup in pickup_neighbors_map[pickup_i]:
        for candidate_idx in list(pickup_index[neighbor_pickup]):
            
            if rides.at[candidate_idx, "is_batchable"]:
                continue
            
            # THE MAGIC HAPPENS HERE: We check against the expanded 1-ring drop zone
            if rides.at[candidate_idx, "drop_hex"] not in drop_neighbors_map[drop_i]:
                continue
                
            # Match Found!
            batch_counter += 1
            rides.at[i, "is_batchable"] = True
            rides.at[candidate_idx, "is_batchable"] = True
            rides.at[i, "batch_id"] = batch_counter
            rides.at[candidate_idx, "batch_id"] = batch_counter
            
            pickup_index[neighbor_pickup].discard(candidate_idx)
            matched = True
            break
            
        if matched:
            break
            
    # 3. If no match, calculate dynamic expiry and add to heap
    if not matched:
        expiry_time = current_time + rides.at[i, "max_wait_mins"]
        heapq.heappush(buffer, (expiry_time, i))
        pickup_index[pickup_i].add(i)

print("Relaxed batching complete!")

Running the relaxed batching engine...
Relaxed batching complete!


In [6]:
# Extract batched rides and join original coordinates
batchable = rides[rides["is_batchable"]].sort_values(by=["batch_id", "t_bin"]).reset_index(drop=True)

cols_to_pull = ["trip_start_lat", "trip_start_lon", "trip_end_lat", "trip_end_lon"]
batchable = batchable.join(df[cols_to_pull], on="orig_idx")
batchable.to_csv(OUTPUT_PATH, index=False)

# --- EVALUATION REPORT CARD ---
print("\n" + "="*40)
print("BATCHING ALGORITHM RESULTS")
print("="*40)

# 1. Efficiency Metrics
total_rides = len(df)
batched_rides = len(batchable)
match_rate = (batched_rides / total_rides) * 100

total_distinct_batches = batchable["batch_id"].nunique()
trips_saved = batched_rides - total_distinct_batches
dispatch_reduction = (trips_saved / total_rides) * 100

print(f"Total Rides Processed:  {total_rides:,}")
print(f"Rides Batched:          {batched_rides:,}")
print(f"Match Rate:             {match_rate:.2f}%")
print(f"Vehicle Trips Saved:    {trips_saved:,}")
print(f"Dispatch Reduction:     {dispatch_reduction:.2f}%")

# 2. Wait Time Metrics (ONLY for rides that were successfully batched)
print("\n--- Wait Time Analysis (Batched Rides Only) ---")
print("ALLOWED Max Wait Time Distribution:")
print(batchable["max_wait_mins"].value_counts().sort_index())

# Calculate actual wait time for batches (Time of last request - Time of first request)
batch_times = batchable.groupby("batch_id")["t_bin"].agg(["min", "max"])
actual_waits = batch_times["max"] - batch_times["min"]

print(f"\nAverage Allowed Wait Time: {batchable['max_wait_mins'].mean():.2f} mins")
print(f"Average Actual Wait Time:  {actual_waits.mean():.2f} mins")

print("\nACTUAL Wait Time Distribution (first person in batch):")
print(actual_waits.value_counts().sort_index())


BATCHING ALGORITHM RESULTS
Total Rides Processed:  1,657,148
Rides Batched:          56,470
Match Rate:             3.41%
Vehicle Trips Saved:    28,235
Dispatch Reduction:     1.70%

--- Wait Time Analysis (Batched Rides Only) ---
ALLOWED Max Wait Time Distribution:
max_wait_mins
3    54122
4     1726
5      422
6      163
7       28
8        9
Name: count, dtype: int64

Average Allowed Wait Time: 3.06 mins
Average Actual Wait Time:  1.67 mins

ACTUAL Wait Time Distribution (first person in batch):
0    4576
1    8204
2    7651
3    7523
4     228
5      36
6      11
7       5
8       1
Name: count, dtype: int64
